# Explore Buildings Data

Do the inspection lat/lon coords line up with buildings?

In [1]:
import os
import sys
import folium
import numpy as np
import pandas as pd
import geopandas as gpd

import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.colors as colors

sys.path.append("../utils")

import config

### Import SB buildings data

In [2]:
sb_buildings_path = os.path.join(
    config.data_dir, "microsoft_buildings", "sb_buildings.geojson"
)

buildings = gpd.read_file(sb_buildings_path)

### Check whether inspection points line up with buildings

#### Import inspections

In [3]:
# Function to read in inspections data for 2019-2023
def import_inspections_data(year):
    """
    Function to read in inspections data for a given year and convert to Albers CRS.
    """
    inspections = os.path.join(
        config.cleaned_inspections_dir, f"inspections_{year}.geojson"
    )
    inspections = gpd.read_file(inspections).to_crs(config.albers_crs)
    return inspections


# Read in inspection data for 2019-2023 as separate dataframes
inspections_dict = {}
for year in range(2019, 2024):
    inspections_dict[year] = import_inspections_data(year)

In [4]:
# Unpack from dictionary
inspections_2019 = inspections_dict[2019]
inspections_2020 = inspections_dict[2020]
inspections_2021 = inspections_dict[2021]
inspections_2022 = inspections_dict[2022]
inspections_2023 = inspections_dict[2023]

#### Count number of inspections that fall outside of buildings

In [5]:
# Create a function that returns the number of inspections that fall outside building geometries for a given year
def inspections_within_buildings(year):
    """
    Function to return the number of inspections that fall within building geometries for a given year.
    """
    inspections = import_inspections_data(year)
    intersects = inspections.sjoin(buildings, predicate="intersects")
    
    return intersects, inspections.loc[~inspections.index.isin(intersects.index)]

# Get the number of inspections that fall within building geometries for each year
inspections_2019, inspections_2019_buildings_false = inspections_within_buildings(2019)
inspections_2020, inspections_2020_buildings_false = inspections_within_buildings(2020)
inspections_2021, inspections_2021_buildings_false = inspections_within_buildings(2021)
inspections_2022, inspections_2022_buildings_false = inspections_within_buildings(2022)
inspections_2023, inspections_2023_buildings_false = inspections_within_buildings(2023)


In [6]:
# For loop to print number of inspections that fall within and outside building geometries for each year
for year in range(2019, 2024):
    inspections_within, inspections_outside = inspections_within_buildings(year)
    print(f"{year} Inspections: \nWithin Buildings - {inspections_within.shape[0]} \nOutside of Buildings - {inspections_outside.shape[0]}\n")

2019 Inspections: 
Within Buildings - 11011 
Outside of Buildings - 3664

2020 Inspections: 
Within Buildings - 9528 
Outside of Buildings - 2350

2021 Inspections: 
Within Buildings - 10491 
Outside of Buildings - 3521

2022 Inspections: 
Within Buildings - 10157 
Outside of Buildings - 3485

2023 Inspections: 
Within Buildings - 9919 
Outside of Buildings - 3454



#### Count number of non-compliant inspections that fall outside of buildings

In [7]:
# For loop to print number of non-compliant inspections that fall within and outside building geometries for each year
for year in range(2019, 2024):
    non_comp_inspections_in, non_comp_inspections_out = inspections_within_buildings(year)
    non_comp_inspections_in = non_comp_inspections_in[non_comp_inspections_in["status"] == "Non-Compliant"]
    non_comp_inspections_out = non_comp_inspections_out[non_comp_inspections_out["status"] == "Non-Compliant"]
    print(f"{year} Non-Compliant Inspections: \nWithin Buildings - {non_comp_inspections_in.shape[0]} \nOutside of Buildings - {non_comp_inspections_out.shape[0]}\n")

2019 Non-Compliant Inspections: 
Within Buildings - 72 
Outside of Buildings - 51

2020 Non-Compliant Inspections: 
Within Buildings - 74 
Outside of Buildings - 20

2021 Non-Compliant Inspections: 
Within Buildings - 63 
Outside of Buildings - 7

2022 Non-Compliant Inspections: 
Within Buildings - 47 
Outside of Buildings - 13

2023 Non-Compliant Inspections: 
Within Buildings - 54 
Outside of Buildings - 17



In [11]:
# Create a function that makes dataframes containing the number of non-compliant inspections that fall outside buildings for each year
def non_comp_inspections_outside(year):
    """
    Function to return the number of non-compliant inspections that fall outside building geometries for a given year.
    """
    inspections = import_inspections_data(year)
    intersects = inspections.sjoin(buildings, predicate="intersects")
    
    non_comp_intersects = intersects[intersects["status"] == "Non-Compliant"]
    non_comp_outside = inspections.loc[~inspections.index.isin(non_comp_intersects.index)]
    non_comp_outside = non_comp_outside[non_comp_outside["status"] == "Non-Compliant"]
    
    return non_comp_intersects, non_comp_outside
# Get the number of non-compliant inspections that fall outside building geometries for each year
non_comp_2019, non_comp_2019_buildings_false = non_comp_inspections_outside(2019)
non_comp_2020, non_comp_2020_buildings_false = non_comp_inspections_outside(2020)
non_comp_2021, non_comp_2021_buildings_false = non_comp_inspections_outside(2021)
non_comp_2022, non_comp_2022_buildings_false = non_comp_inspections_outside(2022)
non_comp_2023, non_comp_2023_buildings_false = non_comp_inspections_outside(2023)


### Calculate distance between non-compliant inspections outside of buildings and nearest building

In [17]:
from shapely.geometry import Point
# Assuming you have two GeoDataFrames: gdf1 and gdf2
gdf1 = non_comp_2019_buildings_false
gdf2 = buildings

# Create a Spatial Index for faster search
sindex = gdf2.sindex

# Find the nearest geometry in gdf2 for each geometry in gdf1
gdf1['nearest_geometry'] = gdf1.geometry.apply(lambda x: gdf2.iloc[sindex.nearest(x)[1][0]].geometry)

# Calculate the distance to the nearest geometry
gdf1['distance_to_nearest'] = gdf1.apply(lambda row: row.geometry.distance(row.nearest_geometry), axis=1)

# Assign inspections to buildings if the distance is less than the threshold
assigned_inspections = inspections[inspections['distance_to_nearest'] < distance_threshold]

# Print the distances, sorted by lowest distance to highest
print(gdf1[['distance_to_nearest']].sort_values(by='distance_to_nearest'))


       distance_to_nearest
14010             0.088252
6807              0.261630
4262              0.761728
6505              1.236091
232               1.347474
10635             1.421246
14121             2.903178
79                2.942639
13992             3.207253
14002             4.836522
14127             5.515628
14012             6.082316
13974             7.032350
13568             7.301402
7                 8.227747
6006              9.544872
785              10.742248
796              11.683629
14669            12.651916
132              14.862070
1612             16.812804
14268            20.548801
14001            26.998716
9755             27.011166
12534            30.804734
136              30.895831
14026            31.491856
14270            31.840457
9414             35.188278
12594            37.083201
4381             40.032028
4378             44.462190
14273            46.150290
13849            47.419364
14020            50.319845
14266            54.432034
1

The minimum distance between buildings in CA is 10 feet (3.048 m). For this analysis, we will assign non-compliant inspections to the closest building within 3 meters. 

In [ ]:
# Assign inspections to buildings if the distance between them is < 3 meters
def assign_inspections_to_buildings(inspections, buildings, distance_threshold=3):
    """
    Function to assign inspections to buildings if the distance between them is less than the distance threshold.
    """
    # Create a Spatial Index for faster search
    sindex = buildings.sindex

    # Find the nearest geometry in buildings for each geometry in inspections
    inspections['nearest_geometry'] = inspections.geometry.apply(lambda x: buildings.iloc[sindex.nearest(x)[1][0]].geometry)

    # Calculate the distance to the nearest geometry
    inspections['distance_to_nearest'] = inspections.apply(lambda row: row.geometry.distance(row.nearest_geometry), axis=1)

    # Assign inspections to buildings if the distance is less than the threshold
    assigned_inspections = inspections[inspections['distance_to_nearest'] < distance_threshold]

    return assigned_inspections

assigned_inspections_2019 = assign_inspections_to_buildings(non_comp_2019_buildings_false, buildings)

In [21]:
assigned_inspections_2019

,fulcrum_id,created_at,updated_at,system_cre,system_upd,version,status,project,assigned_t,latitude,...,editedby,textfield1,textfield2,numberfiel,numberfi_1,core_with_,Date,geometry,nearest_geometry,distance_to_nearest
79,6c5977d8-862a-44de-b48b-1cba952996c0,2018-05-25 16:25:26,2019-12-20 11:38:02,2019-05-07 11:53:05,2019-12-20 11:38:03,5.0,Non-Compliant,None,None,34.670765,...,None,None,None,0.0,0.0,None,2019-12-11,POINT (-7604.035 -371734.546),"POLYGON ((-7607.553 -371718.204, -7621.749 -37...",2.942639
232,e9efa59c-2e32-45ca-8a82-f6ceadf9096b,2018-05-25 16:37:13,2019-12-20 11:38:08,2019-05-07 11:53:54,2019-12-20 11:38:08,5.0,Non-Compliant,None,None,34.674883,...,None,None,None,0.0,0.0,None,2019-12-11,POINT (-7658.936 -371277.398),"POLYGON ((-7662.315 -371286.039, -7659.188 -37...",1.347474
4262,e99e04d5-0050-436e-98b6-e10310e4d3ba,2018-05-25 17:04:03,2019-11-19 14:28:48,2019-05-28 15:30:57,2019-11-19 15:17:48,10.0,Non-Compliant,None,None,34.618535,...,None,None,None,0.0,0.0,None,2019-06-29,POINT (-7142.52 -377532.213),"POLYGON ((-7130.41 -377519.348, -7149.291 -377...",0.761728
6505,be045488-2e2c-471c-8d4c-9d8fceea8293,2018-05-25 16:25:53,2019-08-16 16:21:25,2019-05-28 15:54:24,2019-08-16 16:44:35,2.0,Non-Compliant,None,None,34.647733,...,None,None,None,0.0,0.0,None,2019-08-16,POINT (2250.621 -374294.092),"POLYGON ((2251.227 -374283.406, 2242.43 -37428...",1.236091
6807,3bee5f6f-1c4e-47a4-8b25-d648c55d5ac6,2018-05-25 16:26:44,2019-11-15 17:09:02,2019-05-28 15:55:37,2019-11-15 17:09:05,3.0,Non-Compliant,None,None,34.676359,...,None,None,None,0.0,0.0,None,2019-07-24,POINT (-34995.676 -371046.413),"POLYGON ((-34997.231 -371049.147, -34994.921 -...",0.261630
10635,305df24d-168b-4625-96c1-144770489b8c,2018-05-25 16:39:38,2019-12-11 15:27:32,2019-05-28 16:16:35,2019-12-12 09:56:15,3.0,Non-Compliant,None,None,34.647150,...,None,None,None,0.0,0.0,None,2019-10-10,POINT (-6702.501 -374356.51),"POLYGON ((-6703.852 -374354.132, -6714.389 -37...",1.421246
14010,None,NaT,NaT,NaT,NaT,0.0,Non-Compliant,None,None,34.447430,...,None,None,None,0.0,0.0,None,2019-08-01,POINT (32395.574 -396464.615),"POLYGON ((32376.846 -396467.366, 32391.08 -396...",0.088252
14121,None,NaT,NaT,NaT,NaT,0.0,Non-Compliant,None,None,34.442377,...,None,None,None,0.0,0.0,None,2019-08-01,POINT (31120.357 -397030.048),"POLYGON ((31116.623 -397038.929, 31118.62 -397...",2.903178


## *Create interactive map

***CAUTION: Exercise judgement before running!! Uses too much memory currently.**

In [ ]:
# # Create an interactive map
# m = buildings.explore(
#     color="blue",
#     name="Buildings",
#     tooltip=False,
#     style_kwds={"fillOpacity": 0.2, "weight": 0.5},
#     tiles="OpenStreetMap",
# )

# # Use a subset of columns for tooltips
# tooltip_columns = ["status", "address_fu"] 


# # Add external inspections
# m = inspections_2019_buildings_false.explore(
#     m=m,
#     color="red",
#     marker_kwds={"radius": 1},
#     name="Inspections outside buildings",
#     tooltip=tooltip_columns, 
# )

# # Add layer control
# folium.LayerControl().add_to(m)

# # Display the map
# m